# Lead Scoring Model Comparison Metrics

This notebook compares model metrics from the existing training notebook outputs and identifies the best-performing approach.

In [ ]:
import json
import re
from pathlib import Path

NOTEBOOK_PATH = Path("Lead Scoring Classification Model with 92% Accuracy using RandomForest.ipynb")
nb = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
print(f"Loaded notebook: {NOTEBOOK_PATH.name}")
print(f"Total cells: {len(nb['cells'])}")



In [ ]:
# Parse cross-validation accuracy output
cv_scores = {}
for cell in nb["cells"]:
    for out in cell.get("outputs", []):
        txt = ""
        if "text" in out:
            txt = "".join(out["text"])
        elif "data" in out and "text/plain" in out["data"]:
            tp = out["data"]["text/plain"]
            txt = "".join(tp) if isinstance(tp, list) else str(tp)

        if "RandomForest :" in txt and "GradientBoosting :" in txt:
            pairs = re.findall(r"([A-Za-z ]+?)\s*:\s*([0-9.]+)", txt)
            for name, val in pairs:
                name = name.strip()
                cv_scores[name] = float(val)

print("Cross-validation models found:", len(cv_scores))
for k,v in cv_scores.items():
    print(f"{k}: {v:.6f}")



In [ ]:
# Parse train/test accuracy from model evaluation outputs
current_model = None
metrics = {}

for cell in nb["cells"]:
    if cell["cell_type"] == "markdown":
        md = "".join(cell.get("source", "")).strip().upper()
        if "RANDOM FOREST CLASSIFIER" in md:
            current_model = "RandomForest"
        elif "GRADIENT BOOSTING CLASSIFIER" in md:
            current_model = "GradientBoosting"
        elif "LIGHTGBM CLASSIFIER" in md:
            current_model = "LightGBM"
        elif "CATBOOST CLASSIFIER" in md:
            current_model = "CatBoost"

    if cell["cell_type"] == "code":
        for out in cell.get("outputs", []):
            txt = ""
            if "text" in out:
                txt = "".join(out["text"])
            elif "data" in out and "text/plain" in out["data"]:
                tp = out["data"]["text/plain"]
                txt = "".join(tp) if isinstance(tp, list) else str(tp)

            if "Train Accuracy is:" in txt and "Test Accuracy is:" in txt and current_model:
                m_train = re.search(r"Train Accuracy is:\s*([0-9.]+)", txt)
                m_test = re.search(r"Test Accuracy is:\s*([0-9.]+)", txt)
                if m_train and m_test:
                    metrics[current_model] = {
                        "train_accuracy": float(m_train.group(1)),
                        "test_accuracy": float(m_test.group(1)),
                    }

            # Tuned RandomForest output in this notebook appears as: Train Accuracy: X Test Accuracy: Y
            if "Train Accuracy:" in txt and "Test Accuracy:" in txt:
                m_train2 = re.search(r"Train Accuracy:\s*([0-9.]+)", txt)
                m_test2 = re.search(r"Test Accuracy:\s*([0-9.]+)", txt)
                if m_train2 and m_test2:
                    metrics["Tuned RandomForest"] = {
                        "train_accuracy": float(m_train2.group(1)),
                        "test_accuracy": float(m_test2.group(1)),
                    }

print("Model accuracy blocks parsed:", len(metrics))
for model, vals in metrics.items():
    print(model, vals)



In [ ]:
# Build final comparison table (pure Python, no pandas required)
all_models = sorted(set(list(metrics.keys()) + list(cv_scores.keys())))
rows = []
for m in all_models:
    row = {
        "model": m,
        "cv_accuracy": cv_scores.get(m),
        "train_accuracy": metrics.get(m, {}).get("train_accuracy"),
        "test_accuracy": metrics.get(m, {}).get("test_accuracy"),
    }
    rows.append(row)

# Sort by test accuracy (descending), then CV accuracy
rows_sorted = sorted(rows, key=lambda r: ((r["test_accuracy"] if r["test_accuracy"] is not None else -1), (r["cv_accuracy"] if r["cv_accuracy"] is not None else -1)), reverse=True)

print("\n=== MODEL COMPARISON TABLE ===")
print(f"{'Model':<24} {'CV Acc':<10} {'Train Acc':<12} {'Test Acc':<12}")
print("-"*62)
for r in rows_sorted:
    cv = f"{r['cv_accuracy']:.4f}" if r['cv_accuracy'] is not None else "-"
    tr = f"{r['train_accuracy']:.4f}" if r['train_accuracy'] is not None else "-"
    te = f"{r['test_accuracy']:.4f}" if r['test_accuracy'] is not None else "-"
    print(f"{r['model']:<24} {cv:<10} {tr:<12} {te:<12}")

best = next((r for r in rows_sorted if r['test_accuracy'] is not None), None)
if best:
    print("\nBest model by Test Accuracy:", best['model'])
    print("Best Test Accuracy:", f"{best['test_accuracy']:.4f}")

